[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/convection/convection.ipynb)

# Rayleigh–Bénard Convection

This example computes Rayleigh–Bénard convection: a layer of liquid between a warm floor and a cool ceiling, which conducts heat up while it stays still and turns over in rolls once the warm liquid's buoyancy beats its viscosity and its conduction of heat. Warm liquid is lighter, so the still layer is top-heavy; a parcel that rises keeps its buoyancy only if it rises before its warmth has conducted away, and the Rayleigh number compares the two times. Here the layer turns over from a slightly disturbed, conducting start, and a sweep of Rayleigh numbers finds where a small disturbance stops dying and starts to grow, as [Bergé and Dubois (1984)](https://doi.org/10.1080/00107518408210730) set out.

The algebra is planar spacetime joined with two degenerate directions, one for volume and one for entropy, `Algebra("x+y+t-") * Algebra("v0s0")`. A parcel of the liquid is modelled as a point weighed by its volume, offset along time by its mass and along the entropy's direction by its entropy, `volume + mass * time + heat * entropy`, and a unit volume holds a little less mass the more entropy it holds. The drive of the potentials, minus their gradient wedged with what they act on, pushes a parcel by its commutator with it: the pressure, wedged with volume, pushes its volume, and gravity's potential, wedged with time, pulls its mass, so that a warm parcel's buoyancy is `gravity.commutator(warm - cool)`, gravity pulling on mass alone. The temperature's drive, wedged with entropy, conducts the entropy but pushes nothing: by the Gibbs–Duhem relation a parcel's volume already feels everything its mass and its entropy would. The flow is a stream function on the mesh's vertices; its mass resists its change and its viscosity its turning, and the pressure, which only keeps the flow from filling or emptying any cell, does no work on it and drops out. Mass along time is at once what gravity pulls on and what resists the flow's change.

In the notation of the Oberbeck–Boussinesq equations the layer obeys $\partial_t\mathbf v = -\nabla p/\rho_0 + \nu\nabla^2\mathbf v + g\alpha T\,\hat{\mathbf z}$, $\nabla\cdot\mathbf v = 0$ and $\partial_t T + \mathbf v\cdot\nabla T = \kappa\nabla^2 T$, without the momentum's own advection here, which the large Prandtl number $\nu/\kappa$ of water and oils makes small; the Rayleigh number is $\mathrm{Ra} = g\alpha\,\Delta T\,d^3/(\nu\kappa)$, and between rigid plates the still layer gives way at $\mathrm{Ra}_c \approx 1707$, to rolls of wavenumber $a_c \approx 3.117$ times the inverse depth.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext
from numga.sparse import spdiag
from examples.util.animation import show_animation
from examples.fluids.convection import render, scenarios
from examples.fluids.convection.core import layer

# Planar spacetime, and the volume and the entropy a parcel holds: each a direction of its own, degenerate, since
# nothing measures a volume or an entropy against a length or a time.
spacetime = Algebra("x+y+t-")
ga = spacetime * Algebra("v0s0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
Drive = ga.gatype.from_blades("xt yt xv yv xs ys")                            # Bivector+
axis = mv.x.dual()                                                            # [] Point-
up = mv.y.dual()                                                              # [] Point-
time = -mv.t.dual()                                                           # [] Point-
# A unit of volume, the weight of a place, and a unit of entropy.
volume = mv.v.dual()                                                          # [] Point-
entropy = mv.s.dual()                                                         # [] Point-

## The layer

The layer is a strip of triangles one depth high, closed on itself along its length so that it has a floor and a ceiling but no ends. Its vertices are places of unit volume, and an edge that crosses from the last column back to the first reaches a period further along than its ends' places. A corner's hat gradient is the plane through the facing side, along time, volume and entropy, over twice the triangle's area, so that it reads one at the corner. Lengths are in depths, times in the time heat takes to conduct across the layer, and temperatures in the difference between floor and ceiling, a degree.

In [ ]:
def triangles(topology, edges: Point) -> tuple[Scalar, Point, Plane, Scalar]:
    """Each triangle's area `Scalar+[faces]`, the side facing each corner `[corners] Point-[faces]`, each corner's hat
    gradient `[corners] Plane+[faces]`, and each vertex's share of the area around it `Scalar+[vertices]`."""
    sides = topology.boundary[2, 1] * edges[..., None]                       # [corners] Point-[faces]
    # Twice a triangle's area: two of its sides joined, read through time, at unit volume and entropy.
    double_areas = (((sides[..., 0] & sides[..., 1]) & time) & volume) & entropy   # Scalar-[faces]
    areas = double_areas.norm() / 2                                           # Scalar+[faces]
    gradients = (time & volume & entropy & sides) / double_areas[..., None]   # [corners] Plane+[faces]
    return areas, sides, gradients, (~topology.incidence[2, 0] * (areas / 3)).sum(axis=-1)


# The layer, four depths long: its topology, its vertices' places, the seam's shifts, and its floor and ceiling.
topology, places, shifts, floor, ceiling = layer(scenarios.n_along, scenarios.n_across, scenarios.period)
vertices = (volume + axis * places[:, 0] + up * places[:, 1]).field()         # Point-[vertices]
edges = topology.d[0] * vertices + (axis * shifts).field()                    # Point-[edges]
areas, sides, gradients, cells = triangles(topology, edges)
render.draw_layer(topology, vertices, scenarios.n_across, scenarios.period);

The layer's triangles between its warm floor, red, and its cool ceiling, blue.

## Turning over

The liquid's flow is a stream function, whose rise at a corner carries the liquid along the facing side. Gravity's drive does work on each flow the stream function can make by pulling on the mass the flow carries, `pull & (flows & parcels)`, and only a warm parcel's missing mass is left to pull on, since the still layer's weight is held by the pressure. The flow's kinetic energy is the stiffness of its stream function and its viscous resistance the squared vorticity, so each step solves for the stream function against its mass and its viscosity at once; the flow then carries the entropy along, and conduction spreads it. The liquid has water's Prandtl number, its viscosity seven times its conduction of heat, and the layer starts still and conducting, with a small disturbance. This cell takes a few seconds, most of it stepping the liquid and drawing the animation.

In [ ]:
# The liquid: its density, its viscosity, the Prandtl number in these units, the mass a unit volume loses per degree,
# and gravity, which with them makes the Rayleigh number.
density, viscosity, expansion = scenarios.density, scenarios.prandtl, scenarios.expansion
gravity = scenarios.rayleigh * viscosity / expansion
def convecting(topology, areas: Scalar, sides: Point, gradients: Plane, cells: Scalar, gravity: np.ndarray,
               temperature: Scalar, dt: float, n_steps: int, stride: int) -> Iterator[tuple[Scalar, Scalar]]:
    """A layer's stream function and temperature `[...] Scalar+[vertices]` every `stride` steps of `dt`, from rest
    and this temperature, under each gravity `[...]`, the stream function nothing and the temperature held at the
    floor and the ceiling."""
    corners = topology.incidence[2, 0]                                        # [corners] Scalar+[faces <- vertices]
    inside = topology.reading(topology.interior[0])                           # Scalar+[free <- vertices]
    # The stream function's kinetic form, which is also the conduction of heat, gathered from each triangle's corners
    # onto the vertices they join, and its viscous form.
    local = (gradients[..., :, None] | gradients[..., None, :]) * areas[..., None, None]   # [corners, corners] Scalar+[faces]
    stiffness = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=-1).sum(axis=-1)
    viscous = viscosity * stiffness * spdiag(1 / cells) * stiffness           # Scalar+[vertices <- vertices]
    # Each step's implicit maps on the free vertices: the flow's inertia with its viscosity, and the cells' heat with
    # its conduction.
    flowing = (inside * (density / dt * stiffness + viscous) * ~inside) * Scalar
    warming = (inside * (spdiag(cells / dt) + stiffness) * ~inside) * Scalar
    # A stream function rising by one at a corner carries half the facing side's worth of liquid over the triangle.
    flows = sides / 2                                                         # [corners] Point-[faces]
    pull = -(gravity * mv.yt)                                                 # [...] Drive+
    stream = 0 * temperature                                                  # [...] Scalar+[vertices]
    for step in range(n_steps + 1):
        if step % stride == 0:
            yield stream, temperature
        # Each triangle's parcels per volume: unit volume, holding less mass the warmer they are, and their entropy.
        warmth = corners.mean(axis=0) * temperature                           # [...] Scalar+[faces]
        parcels = volume + (density - expansion * warmth) * time + warmth * entropy   # [...] Point-[faces]
        load = (~corners * (pull[..., None] & (flows & parcels[..., None]))).sum(axis=-1)   # [...] Scalar+[vertices]
        stream = ~inside * flowing.solve(inside * (density / dt * (stiffness * stream) + load))
        # The flow carries the entropy along; what the flux carries out of each corner's cell leaves it.
        velocities = ((corners * stream[..., None]) * sides).sum(axis=-1) / (2 * areas)   # [...] Point-[faces]
        flux = velocities.regressive(warmth * entropy)                        # [...] Flux-[faces]
        carried = mv.s & (~corners * (areas * (gradients ^ flux[..., None]))).sum(axis=-1)   # [...] Scalar+[vertices]
        held = temperature - ~inside * (inside * temperature)                 # [...] Scalar+[vertices]
        temperature = held + ~inside * warming.solve(inside * (cells / dt * temperature - carried - stiffness * held))


start = mv.scalar(scenarios.conducting(places, scenarios.disturbance)[:, None]).field()   # Scalar+[vertices]
# Shown once the rolls begin to show.
frames = list(convecting(topology, areas, sides, gradients, cells, np.array(gravity), start, scenarios.dt,
                         scenarios.n_steps, scenarios.stride))[scenarios.n_unshown // scenarios.stride:]
show_animation(render.animate(topology, vertices, scenarios.n_across, scenarios.period, frames), "convection",
               scenarios.frame_duration_ms)

The layer is coloured by its temperature, red above the middle's and blue below, with the streamlines in black, spaced in each frame by its strongest flow. The animation starts as the disturbance's rolls begin to show in the streamlines, while the layer still conducts. The rolls grow, warm liquid rising and cool liquid sinking between them, and overshoot: they turn the warm liquid right over, mixing the layer's middle until the top-heaviness that drives them is spent, and nearly stall. Conduction builds the warm layer at the floor and the cool one at the ceiling up again, the rolls pick up, and after a few swings they carry the heat up steadily; the temperature then changes only in thin layers at the floor and the ceiling, where conduction hands the heat to the flow and takes it back.

## The onset

A small disturbance of the still layer grows or dies at a steady rate once its other parts have died away, and the rate's sign tells whether the layer is stable. The sweep runs the layer at several Rayleigh numbers at once, a batch of gravities, on a layer one wavelength of the first rolls long; the flow's maps do not depend on gravity, so every case shares their factorizations. The growth rate is read from the stream function's size at two times. This cell takes about five seconds.

In [ ]:
# The layer one wavelength of the first rolls long, at these Rayleigh numbers.
rayleighs = scenarios.onset_rayleighs                                         # [rayleighs]
onset_topology, onset_places, onset_shifts, _, _ = layer(scenarios.onset_along, scenarios.n_across,
                                                          scenarios.onset_period)
onset_vertices = (volume + axis * onset_places[:, 0] + up * onset_places[:, 1]).field()   # Point-[vertices]
onset_edges = onset_topology.d[0] * onset_vertices + (axis * onset_shifts).field()          # Point-[edges]
onset_areas, onset_sides, onset_gradients, onset_cells = triangles(onset_topology, onset_edges)
# The still layer, disturbed by a roll a wavelength along and half a wave up.
rolls = 1e-4 * np.sin(2 * np.pi * onset_places[:, 0] / scenarios.onset_period) * np.sin(np.pi * onset_places[:, 1])
disturbed = mv.scalar((scenarios.conducting(onset_places, 0.0) + rolls)[:, None]).field()   # Scalar+[vertices]
readings = list(convecting(onset_topology, onset_areas, onset_sides, onset_gradients, onset_cells,
                           rayleighs * viscosity / expansion, disturbed, scenarios.onset_dt, scenarios.onset_steps,
                           scenarios.onset_stride))
# The stream function's squared size over the layer at the middle and at the end; it grows at twice the rate.
sizes = [(stream * onset_cells * stream).sites.sum() for stream, _ in readings[1:]]   # [2] [rayleighs] Scalar+
growth = (sizes[1].log() - sizes[0].log()) / (2 * scenarios.onset_stride * scenarios.onset_dt)   # [rayleighs] Scalar+
render.draw_onset(rayleighs, growth);

The growth rate against the Rayleigh number crosses zero at the dotted line, the onset between rigid plates: below it a disturbance dies, above it the rolls grow.

In [ ]:
# checks
# Gravity pulls a parcel's mass down and passes over its volume and its entropy.
pulled = (mv.y & (-mv.yt).commutator(time + volume + entropy)).to_array()
np.testing.assert_allclose(pulled, -1.0, rtol=1e-12)
# The growth rate crosses zero within one percent of the onset between rigid plates.
rates = growth.to_array()
crossing = np.interp(0.0, rates, rayleighs)
np.testing.assert_allclose(crossing, 1707.8, rtol=0.01)
# Turned over, the layer's middle half is all but even in temperature on average, the drop in the thin layers at the
# floor and the ceiling; conducting, it drops half a degree.
temperatures = frames[-1][1].to_array()
middle_drop = (temperatures[np.isclose(places[:, 1], 0.25)].mean() - temperatures[np.isclose(places[:, 1], 0.75)].mean())
assert abs(middle_drop) < 0.05